# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamzanPUCIT/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

---
**Task type: ranking / scoring.**

The editor's question is *"which pages first?"*, not *"yes or no for each page"*. So the output
is a **score per page**, and the pages are sorted by it. Under the hood I can get that score from a
classifier's probability (will this page lose demand?), but what I evaluate is the **order** of
the list, because the editor only ever reads the top of it.

Frame in one paragraph (from `framing-ml-problems`):
> For a client's content editor, deciding which pages to refresh first, we will build a ranked
> review queue from the starter content snapshot, scoring each page's risk of a demand drop
> (impressions in the last 30 days falling > 20% below the 30 days before), measured by
> precision@K against the base rate. A wrong call costs a wasted review (false alarm) or a
> high-demand page that keeps sliding (miss). A plain rule isn't enough because the drop rate
> depends on several signals at once and differs a lot by client. We will claim only observed,
> directional, decision-support results.

In [1]:
# Setup: load the starter dataset (works in a local clone and in Colab)
from pathlib import Path
import numpy as np
import pandas as pd

RAW_URL = ("https://raw.githubusercontent.com/RamzanPUCIT/flyrank-ml-internship/"
           "main/data/raw/content_refresh_anonymized.csv")
local = [p for p in [Path("../../data/raw/content_refresh_anonymized.csv"),
                     Path("data/raw/content_refresh_anonymized.csv")] if p.exists()]
df = pd.read_csv(local[0] if local else RAW_URL)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("rows x cols:", df.shape)

# My lane slice (same definition in every notebook)
MIN_PREV_IMPR = 100          # page needs real demand in the earlier window
lane = df[df["impressions_prev_30d"] >= MIN_PREV_IMPR].copy()
# proxy target: observed later-window outcome (last 30d vs the 30d before)
lane["demand_drop"] = (lane["impressions_last_30d"] < 0.8 * lane["impressions_prev_30d"]).astype(int)

# The output shape of a ranking task: one score per page, sorted.
# (placeholder score for now = random, just to show the shape; the real one comes in week 4)
rng = np.random.default_rng(42)
demo = lane[["content_id", "client_id"]].copy()
demo["score"] = rng.random(len(demo))
demo.sort_values("score", ascending=False).head(5)

rows x cols: (30000, 44)


,content_id,client_id,score
12502,content_0106fc459b18,client_6208ef0f77,0.999977
20401,content_62c74ae619f8,client_4e07408562,0.999947
12358,content_a86137c029a4,client_4e07408562,0.999861
3763,content_99207183c101,client_3fdba35f04,0.999765
8207,content_a5f41d4af8c3,client_4e07408562,0.999715


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

---
**Target (proxy): `demand_drop`** = 1 when a page's impressions in the most recent 30 days are
more than 20% below its impressions in the 30 days before that (`impressions_last_30d <
0.8 × impressions_prev_30d`), for pages with ≥ 100 impressions in that earlier window.

- **Where it comes from:** an **observed outcome** in a *later* time window (days 1–30 back)
  compared with an *earlier* window (days 31–60 back). It is not someone's product flag.
- It matches the starter's `is_declining_label` / `trend_direction == "down"` exactly on my slice
  (checked below) — that is why `trend_direction` and `trend_pct` can never be inputs.
- **It is a proxy**, not the real goal: the real goal is "a refresh would pay off". A drop says
  *look at this page*, not *a refresh will fix it* (seasonality, a lost competitor, or a
  tracking change can also cause a drop).
- The ≥ 100 floor keeps tiny pages out — at 20 impressions a "drop" can be pure noise.

In [2]:
same = (lane["demand_drop"] == (lane["trend_direction"] == "down").astype(int)).mean()
print(f"agreement with trend_direction == 'down' on my slice: {same:.1%}")
print("class balance (share of pages with a demand drop):")
print(lane["demand_drop"].value_counts(normalize=True).round(3))

agreement with trend_direction == 'down' on my slice: 100.0%
class balance (share of pages with a demand drop):
demand_drop
1    0.616
0    0.384
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

---
**Metric: precision@K** — of the top K pages in my queue, what share really had a demand drop?
I will report **K = 50 and K = 100**, on clients the method has not seen, always next to the
**base rate** (what random ordering gets).

- Base rate in my slice is ~62% (cell below), so random picking already gets ~0.62.
- **"Good" = clearly above the base rate**: I will call the queue useful if precision@50 is at
  least **10 points above the base rate** on held-out clients (≈ 0.72+), and a model only "wins"
  if it beats my week-4 rule baseline on the same split.
- I also print ROC-AUC as a secondary check of the whole ordering, but precision@K is the one
  that matches what the editor sees.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())

base_rate = lane["demand_drop"].mean()
rand = [precision_at_k(np.random.default_rng(s).random(len(lane)), lane["demand_drop"].values, 50)
        for s in range(200)]
print(f"base rate: {base_rate:.3f}")
print(f"random ranking precision@50 over 200 seeds: mean {np.mean(rand):.3f}, "
      f"5-95% range {np.percentile(rand, 5):.2f}-{np.percentile(rand, 95):.2f}")
print(f"my 'good' bar for precision@50: >= {base_rate + 0.10:.2f}")

base rate: 0.616
random ranking precision@50 over 200 seeds: mean 0.621, 5-95% range 0.50-0.74
my 'good' bar for precision@50: >= 0.72


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

---
**One row = one content page** (one pseudonymized `content_id`), described by a snapshot of
its trailing-90-day metrics, split into an earlier window (days 31–60 back) and a later window
(days 1–30 back).

- **Inputs** come from the earlier window or are static page properties (age, length, keyword
  context, type, intent).
- **The target** comes from the later window.
- `client_id` stays in the frame only for grouping and for client-held-out splits later.

The dataframe below is the real unit I will rank (18,010 pages from 30 clients), and the grain
check confirms one row per page.

In [4]:
lane["ctr_prev"] = lane["clicks_prev_30d"] / lane["impressions_prev_30d"] * 100  # x100 percent
unit_cols = ["content_id", "client_id",                       # context (never features)
             "impressions_prev_30d", "clicks_prev_30d", "ctr_prev", "sessions_prev_30d",
             "content_age_days", "word_count", "search_volume", "content_type", "main_intent",
             "demand_drop"]                                    # target
unit = lane[unit_cols]
print("rows:", len(unit), "| unique content_id:", unit["content_id"].nunique(),
      "| clients:", unit["client_id"].nunique())
assert unit["content_id"].is_unique, "grain broken: more than one row per page"
unit.head(8)

rows: 18010 | unique content_id: 18010 | clients: 30


,content_id,client_id,impressions_prev_30d,clicks_prev_30d,ctr_prev,sessions_prev_30d,content_age_days,word_count,search_volume,content_type,main_intent,demand_drop
0,content_304f48230142,client_f369cb89fc,987,13,1.317123,9,187,3221.0,10.0,keyword article,transactional,1
1,content_a1fb4e703a9e,client_4e07408562,5915,1,0.016906,2,445,2481.0,90.0,keyword article,informational,1
2,content_9aa793d4d895,client_7f2253d7e2,6089,3,0.049269,3,141,3515.0,0.0,keyword article,informational,1
3,content_331d6c4de07b,client_19581e27de,4206,17,0.404184,26,463,NaN,10.0,keyword article,commercial,0
4,content_d99b7a2d90ca,client_3fdba35f04,6452,2,0.030998,9,263,2803.0,0.0,keyword article,informational,1
5,content_d4084a4bc775,client_f369cb89fc,1009,1,0.099108,1,147,3080.0,720.0,keyword article,transactional,1
7,content_a63219c6e95a,client_19581e27de,632,0,0.000000,4,445,NaN,590.0,keyword article,commercial,0
8,content_5e6c160719bc,client_6208ef0f77,13828,8,0.057854,14,90,3807.0,0.0,keyword article,informational,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

---
A fixed if-statement is a fair starting point (and it will be my week-4 baseline), but the cell
below shows why one rule is not the end:

- **Signals stack.** "Young page" alone and "zero clicks" alone each lift the drop rate a bit;
  together they lift it more — and the effect of one depends on the other (the age × CTR table).
  Writing every combination by hand does not scale past two or three signals.
- **Clients differ a lot.** The drop rate per client goes from 0% to 100%. A threshold tuned on one
  client can be wrong on the next one; a learned score can combine many weak signals and be
  tested on clients it never saw.
- **It shifts over time.** Thresholds that worked last quarter decay; a model can be retrained.

So ML earns its place only if it beats the rule on held-out clients — that comparison is the
whole point of weeks 4–5. If it doesn't, the rule wins and I will say so.

In [5]:
young = lane["content_age_days"] <= 180
zero_click = lane["clicks_prev_30d"] == 0
for name, mask in [("young page (<=180d)", young), ("zero clicks in earlier window", zero_click),
                   ("both", young & zero_click)]:
    print(f"{name:32s} n={mask.sum():6,d}  drop rate={lane.loc[mask, 'demand_drop'].mean():.3f}")
print(f"{'all slice pages':32s} n={len(lane):6,d}  drop rate={lane['demand_drop'].mean():.3f}")

age_b = pd.cut(lane["content_age_days"], [89, 180, 365, 5000], labels=["90-180", "181-365", "365+"])
ctr_b = pd.cut(lane["ctr_prev"], [-0.01, 0, 0.5, 100], labels=["0", "0-0.5%", ">0.5%"])
print("\ndrop rate by age x earlier-window CTR (second table = number of pages per cell):")
display(lane.pivot_table(index=age_b, columns=ctr_b, values="demand_drop", aggfunc="mean", observed=True).round(3))
display(lane.pivot_table(index=age_b, columns=ctr_b, values="demand_drop", aggfunc="size", observed=True))

per_client = lane.groupby("client_id")["demand_drop"].mean()
print(f"drop rate per client: min {per_client.min():.2f}, median {per_client.median():.2f}, max {per_client.max():.2f}")

young page (<=180d)              n= 7,394  drop rate=0.713
zero clicks in earlier window    n= 6,908  drop rate=0.667
both                             n= 2,760  drop rate=0.799
all slice pages                  n=18,010  drop rate=0.616

drop rate by age x earlier-window CTR (second table = number of pages per cell):


ctr_prev,0,0-0.5%,>0.5%
content_age_days,,,
90-180,0.799,0.708,0.552
181-365,0.689,0.597,0.501
365+,0.453,0.469,0.368


ctr_prev,0,0-0.5%,>0.5%
content_age_days,,,
90-180,2760,3253,1381
181-365,2223,3080,1101
365+,1925,1800,487


drop rate per client: min 0.00, median 0.61, max 1.00


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.